In [ ]:
import os
import json
import pandas as pd
import numpy as np

In [ ]:
def get_metadata_into_dataframe(parent_folder):
    
    all_metadata = []

    # Walk through the parent folder and its sub-directories
    for root, dirs, files in os.walk(parent_folder):
        for file in files:
            if file == "metadata.json":
                metadata_file_path = os.path.join(root, file)
                try:
                    with open(metadata_file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        all_metadata.append(data)
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON from {metadata_file_path}: {e}")
                except Exception as e:
                    print(f"An error occurred while reading {metadata_file_path}: {e}")

    if all_metadata:
        # Create a DataFrame from the list of dictionaries
        df = pd.DataFrame(all_metadata)
        return df
    else:
        print(f"No 'metadata.json' files found in '{parent_folder}' or its subfolders.")
        return pd.DataFrame()

In [ ]:
main_folder_path = 'data/OCR_Final' 

In [ ]:
metadata_df = get_metadata_into_dataframe(main_folder_path)

In [5]:
metadata_df.head()

,title,author,genre,issued_date,written_date,ocr_confidence
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,Unknown,Non-Fiction; Religious,1906,1187 - 1225,0.9682
1,නිදහසේ මන්ත්‍රය,ඇස් මහින්ද හිමි,Non-Fiction; Poetry,1938,1901 - 1938,0.8997
2,පැරණි ගම,ගල්පාත ඛේමානන්ද හිමි,Non-Fiction; History,1944,1944,0.9846
3,පන්සිය පනස් ජාතක පොත,Unknown,Fiction; Religious,1881,1303 - 1333,0.9987
4,හිතෝපදේශ සන්නය,වැලිගම ශ්‍රී සුමංගල හිමි,Non-Fiction,1884,1825 - 1905,0.9871


In [6]:
len(metadata_df)

46

In [7]:
split_genres = metadata_df['genre'].str.split(';', expand=True)

In [8]:
metadata_df['genre_main'] = split_genres[0].str.strip()

In [9]:
metadata_df['genre_sub'] = split_genres[1].str.strip().fillna('')

In [10]:
metadata_df.head()

,title,author,genre,issued_date,written_date,ocr_confidence,genre_main,genre_sub
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,Unknown,Non-Fiction; Religious,1906,1187 - 1225,0.9682,Non-Fiction,Religious
1,නිදහසේ මන්ත්‍රය,ඇස් මහින්ද හිමි,Non-Fiction; Poetry,1938,1901 - 1938,0.8997,Non-Fiction,Poetry
2,පැරණි ගම,ගල්පාත ඛේමානන්ද හිමි,Non-Fiction; History,1944,1944,0.9846,Non-Fiction,History
3,පන්සිය පනස් ජාතක පොත,Unknown,Fiction; Religious,1881,1303 - 1333,0.9987,Fiction,Religious
4,හිතෝපදේශ සන්නය,වැලිගම ශ්‍රී සුමංගල හිමි,Non-Fiction,1884,1825 - 1905,0.9871,Non-Fiction,


In [11]:
metadata_df['start_date'] = np.nan
metadata_df['end_date'] = np.nan

In [12]:
for index, row in metadata_df.iterrows():
    date_str = str(row['written_date']) # Ensure it's treated as a string
    
    start_year = np.nan
    end_year = np.nan

    if ' - ' in date_str:
        # It's a range
        try:
            start_year_str, end_year_str = date_str.split(' - ')
            start_year = int(start_year_str.strip())
            end_year = int(end_year_str.strip())
        except ValueError:
            print(f"Warning: Could not parse range '{date_str}' at index {index}. Setting to NaN.")
    else:
        # It's a single year or potentially unparseable
        try:
            single_year = int(date_str.strip())
            start_year = single_year
            end_year = single_year # Duplicate for single date
        except ValueError:
            print(f"Warning: Could not parse single year/date '{date_str}' at index {index}. Setting to NaN.")
            
    # Assign the parsed (or NaN) values back to the DataFrame
    metadata_df.at[index, 'start_date'] = start_year
    metadata_df.at[index, 'end_date'] = end_year


In [13]:
metadata_df.head()

,title,author,genre,issued_date,written_date,ocr_confidence,genre_main,genre_sub,start_date,end_date
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,Unknown,Non-Fiction; Religious,1906,1187 - 1225,0.9682,Non-Fiction,Religious,1187.0,1225.0
1,නිදහසේ මන්ත්‍රය,ඇස් මහින්ද හිමි,Non-Fiction; Poetry,1938,1901 - 1938,0.8997,Non-Fiction,Poetry,1901.0,1938.0
2,පැරණි ගම,ගල්පාත ඛේමානන්ද හිමි,Non-Fiction; History,1944,1944,0.9846,Non-Fiction,History,1944.0,1944.0
3,පන්සිය පනස් ජාතක පොත,Unknown,Fiction; Religious,1881,1303 - 1333,0.9987,Fiction,Religious,1303.0,1333.0
4,හිතෝපදේශ සන්නය,වැලිගම ශ්‍රී සුමංගල හිමි,Non-Fiction,1884,1825 - 1905,0.9871,Non-Fiction,,1825.0,1905.0


In [14]:
metadata_df['genre_main'].value_counts()

genre_main
Non-Fiction    32
Fiction        14
Name: count, dtype: int64

In [15]:
metadata_df['genre_sub'].value_counts()

genre_sub
Religious    18
Poetry       12
Language      8
History       5
Medical       2
              1
Name: count, dtype: int64

In [16]:
def get_century(year):
    """Calculates the century for a given year."""
    if pd.isna(year):
        return np.nan
    return (int(year) - 1) // 100 + 1

In [17]:
metadata_df['end_century'] = metadata_df['end_date'].apply(get_century)

In [18]:
century_genre_main_counts = pd.crosstab(metadata_df['end_century'], metadata_df['genre_main'])

In [19]:
century_genre_main_counts

genre_main,Fiction,Non-Fiction
end_century,,
5,0,1
13,1,7
14,2,2
15,4,1
18,0,3
19,3,6
20,4,12


In [20]:
century_genre_sub_counts = pd.crosstab(metadata_df['end_century'], metadata_df['genre_sub'])

In [21]:
century_genre_sub_counts

genre_sub,,History,Language,Medical,Poetry,Religious
end_century,,,,,,
5,0,0,0,1,0,0
13,0,0,2,0,1,5
14,0,1,0,0,0,3
15,0,0,0,0,3,2
18,0,0,1,1,0,1
19,0,2,2,0,3,2
20,1,2,3,0,5,5


In [22]:
len(metadata_df['author'][metadata_df['author'] != 'Unknown'])

32

In [23]:
len(metadata_df['author'][metadata_df['author'] == 'Unknown'])

14

In [24]:
metadata_df['author'][metadata_df['author'] != 'Unknown'].value_counts().to_frame()

,count
author,
මුනිදාස කුමාරතුංග,4
හික්කඩුවේ ශ්‍රි සුමංගල හිමි,2
මාදම්පේ ධම්මතිලක හිමි,2
ඇස් මහින්ද හිමි,1
ශ්‍රීමද් බුද්ධදාස රජතුමා,1
රත්මලානේ ධර්මාලෝක හිමි,1
දොන් අන්ද්‍රිස් සිල්වා,1
වැලිවිට සරණංකර සංඝරාජ හිමි,1
වටද්දර මේධානන්ද හිමි; සිරි පරාකුමබාහු විල්ගම්මුල සංඝරාජ හිමි,1


In [25]:
metadata_df['ocr_confidence'].describe()

count    46.000000
mean      0.968359
std       0.041145
min       0.855300
25%       0.967375
50%       0.988500
75%       0.996950
max       0.999700
Name: ocr_confidence, dtype: float64

In [41]:
metadata_df.to_csv("metadata.csv", index = False)

In [26]:
page_count = pd.read_csv("page_counts.csv")

In [28]:
page_count.head()

,title,pages
0,අධිමාස දීපනය,7
1,අධිමාස විනිශ්චය,5
2,අධිමාස සංග්‍රහව,5
3,අනාගතවංශය : මෙතේ බුදු සිරිත,6
4,අ‍‍ශෝක ශිලාලිපි සහ ප්‍රතිමාකරණ විනිශ්චය,5


In [42]:
import locale

In [45]:
metadata_df['sort_key'] = metadata_df['title'].apply(locale.strxfrm)

In [46]:
df_sorted = metadata_df.sort_values(by='sort_key').drop(columns='sort_key')

In [54]:
df_sorted

,title,author,genre,issued_date,written_date,ocr_confidence,genre_main,genre_sub,start_date,end_date,end_century,title_prefix
21,අධිමාස දීපනය,මාදම්පේ ධම්මතිලක හිමි,Non-Fiction; Religious,1896,1850 - 1896,0.9984,Non-Fiction,Religious,1850.0,1896.0,19,අධි
26,අධිමාස විනිශ්චය,වැලිකන්දේ ශ්‍රී සුමංගල හිමි,Non-Fiction; Religious,1904,1850 - 1904,0.9971,Non-Fiction,Religious,1850.0,1904.0,20,අධි
5,අධිමාස සංග්‍රහව,මාදම්පේ ධම්මතිලක හිමි,Non-Fiction; Religious,1903,1850 - 1903,0.9692,Non-Fiction,Religious,1850.0,1903.0,20,අධි
32,අනාගතවංශය : මෙතේ බුදු සිරිත,වටද්දර මේධානන්ද හිමි; සිරි පරාකුමබාහු විල්ගම්ම...,Fiction; Religious,1934,1325 - 1333,0.9992,Fiction,Religious,1325.0,1333.0,14,අනා
24,අ‍‍ශෝක ශිලාලිපි සහ ප්‍රතිමාකරණ විනිශ්චය,ඩී. ඊ. වික්‍රමසූරිය,Non-Fiction; History,1919,1916,0.9989,Non-Fiction,History,1916.0,1916.0,20,අ‍‍
34,ඔකඳපොල සන්නය හෙවත් බාලාවතාර ලියන සන්න,දොන් අන්ද්‍රිස් සිල්වා,Non-Fiction; Language,1888,1760 - 1778,0.9884,Non-Fiction,Language,1760.0,1778.0,18,ඔකඳ
13,කාව්‍ය වජ්‍රායුධය - පළමු කොටස,එංගල්තිනා කුමාරි,Fiction; Poetry,1889,1825 - 1893,0.9254,Fiction,Poetry,1825.0,1893.0,19,කාව
20,කාව්‍යශේඛරය,තොටගමුවේ රාහුල හිමි,Fiction; Poetry,1872,1408 - 1491,0.9813,Fiction,Poetry,1408.0,1491.0,15,කාව
35,කුදුසික,Unknown,Non-Fiction; Poetry,1894,1270 - 1293,0.9988,Non-Fiction,Poetry,1270.0,1293.0,13,කුද
39,කුසජාතක විවරණය (ප්‍රථම භාගය),මුනිදාස කුමාරතුංග,Non-Fiction; Religious,1932,1887 - 1932,0.9701,Non-Fiction,Religious,1887.0,1932.0,20,කුස


In [37]:
merged_df = pd.merge(metadata_df, page_count, on='title_prefix', how='left', suffixes=('_pdf', '_meta'))

In [ ]:
df_sorted = df_sorted.reset_index(drop=True)

In [58]:
merged_df_left = pd.merge(df_sorted, page_count, left_index=True, right_index=True, how='left')


In [63]:
len(merged_df_left)

46

In [64]:
merged_df_left[['title_x', 'title_y']]

,title_x,title_y
0,අධිමාස දීපනය,අධිමාස දීපනය
1,අධිමාස විනිශ්චය,අධිමාස විනිශ්චය
2,අධිමාස සංග්‍රහව,අධිමාස සංග්‍රහව
3,අනාගතවංශය : මෙතේ බුදු සිරිත,අනාගතවංශය : මෙතේ බුදු සිරිත
4,අ‍‍ශෝක ශිලාලිපි සහ ප්‍රතිමාකරණ විනිශ්චය,අ‍‍ශෝක ශිලාලිපි සහ ප්‍රතිමාකරණ විනිශ්චය
5,ඔකඳපොල සන්නය හෙවත් බාලාවතාර ලියන සන්න,ඔකඳපොල සන්නය හෙවත් බාලාවතාර ලියන සන්න
6,කාව්‍ය වජ්‍රායුධය - පළමු කොටස,කාව්‍ය වජ්‍රායුධය - පළමු කොටස
7,කාව්‍යශේඛරය,කාව්‍යෙශ්ඛරය
8,කුදුසික,කුදුසික
9,කුසජාතක විවරණය (ප්‍රථම භාගය),කුසජාතක විවරණය (ප්‍රථම භාගය)


In [65]:
merged_df_left.head()

,title_x,author,genre,issued_date,written_date,ocr_confidence,genre_main,genre_sub,start_date,end_date,end_century,title_prefix_x,title_y,pages,title_prefix_y
0,අධිමාස දීපනය,මාදම්පේ ධම්මතිලක හිමි,Non-Fiction; Religious,1896,1850 - 1896,0.9984,Non-Fiction,Religious,1850.0,1896.0,19,අධි,අධිමාස දීපනය,7,අධි
1,අධිමාස විනිශ්චය,වැලිකන්දේ ශ්‍රී සුමංගල හිමි,Non-Fiction; Religious,1904,1850 - 1904,0.9971,Non-Fiction,Religious,1850.0,1904.0,20,අධි,අධිමාස විනිශ්චය,5,අධි
2,අධිමාස සංග්‍රහව,මාදම්පේ ධම්මතිලක හිමි,Non-Fiction; Religious,1903,1850 - 1903,0.9692,Non-Fiction,Religious,1850.0,1903.0,20,අධි,අධිමාස සංග්‍රහව,5,අධි
3,අනාගතවංශය : මෙතේ බුදු සිරිත,වටද්දර මේධානන්ද හිමි; සිරි පරාකුමබාහු විල්ගම්ම...,Fiction; Religious,1934,1325 - 1333,0.9992,Fiction,Religious,1325.0,1333.0,14,අනා,අනාගතවංශය : මෙතේ බුදු සිරිත,6,අනා
4,අ‍‍ශෝක ශිලාලිපි සහ ප්‍රතිමාකරණ විනිශ්චය,ඩී. ඊ. වික්‍රමසූරිය,Non-Fiction; History,1919,1916,0.9989,Non-Fiction,History,1916.0,1916.0,20,අ‍‍,අ‍‍ශෝක ශිලාලිපි සහ ප්‍රතිමාකරණ විනිශ්චය,5,අ‍‍


In [66]:
pd.crosstab(merged_df_left['end_century'], merged_df_left['pages'])

pages,5,6,7,8
end_century,,,,
5,1,0,0,0
13,3,3,2,0
14,2,2,0,0
15,4,0,1,0
18,2,1,0,0
19,5,3,1,0
20,10,4,1,1


In [67]:
merged_df_left['end_date'].describe()

count      46.000000
mean     1654.782609
std       333.425315
min       426.000000
25%      1347.250000
50%      1847.500000
75%      1911.000000
max      1944.000000
Name: end_date, dtype: float64